# 5. Submission Generation

This notebook generates the final submission file for the BirdClef 2026 competition using the ensemble predictions.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import json
import torchaudio
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('/home/kwierman/Desktop/Projects/BirdClef')
MODELS_DIR = BASE_DIR / 'notebooks' / 'models'
TEST_SOUNDSCAPES_DIR = BASE_DIR / 'test_soundscapes'
SAMPLE_SUBMISSION_CSV = BASE_DIR / 'sample_submission.csv'
OUTPUT_DIR = BASE_DIR / 'notebooks' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 5.1 Load Configuration and Models

In [ ]:
# Load configuration
with open(MODELS_DIR / 'ensemble_config.json', 'r') as f:
    config = json.load(f)

num_classes = config['num_classes']
species_list = config['species_list']
training_results = config['training_results']

# Load sample submission to get exact format
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)
print(f"Sample submission shape: {sample_sub.shape}")
print(f"Columns (first 5): {sample_sub.columns[:5].tolist()}")
print(f"Columns (last 5): {sample_sub.columns[-5:].tolist()}")

In [ ]:
# Model definitions (same as in prediction notebook)
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 512, stride=160), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Sequential(nn.Linear(256, 512), nn.ReLU(), nn.Dropout(0.4),
                                        nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
                                        nn.Linear(256, num_classes))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.features(x).squeeze(-1)
        return self.classifier(x)


class LSTMModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.input_proj = nn.Linear(128, 64)
        self.lstm = nn.LSTM(64, 256, 2, batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.4),
                                        nn.Linear(256, num_classes))
    def forward(self, x):
        x = torch.stft(x.squeeze(1), n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)[:, :, :128]
        x = self.input_proj(x)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])


class ResNet1D(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2)
        self.layer3 = self._make_layer(128, 256, 2)
        self.layer4 = self._make_layer(256, 512, 2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_ch, out_ch, blocks):
        layers = [nn.Sequential(nn.Conv1d(in_ch, out_ch, 3, stride=2 if in_ch != out_ch else 1, padding=1),
                               nn.BatchNorm1d(out_ch), nn.ReLU())]
        for _ in range(1, blocks):
            layers.append(nn.Sequential(nn.Conv1d(out_ch, out_ch, 3, padding=1),
                                       nn.BatchNorm1d(out_ch), nn.ReLU()))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).view(x.size(0), -1)
        return self.classifier(x)


class TransformerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.input_proj = nn.Linear(128, 256)
        self.pos_encoder = nn.Sequential(
            nn.Embedding(5000, 256),
        )
        encoder_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=512, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, 4)
        self.classifier = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.3),
                                        nn.Linear(256, num_classes))
    
    def forward(self, x):
        x = torch.stft(x.squeeze(1), n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)[:, :, :128]
        x = self.input_proj(x)
        x = self.transformer(x).mean(dim=1)
        return self.classifier(x)

# Load models
models = {}
for name in ['cnn', 'lstm', 'resnet', 'transformer']:
    model_class = {'cnn': CNNModel, 'lstm': LSTMModel, 'resnet': ResNet1D, 'transformer': TransformerModel}[name]
    model = model_class(num_classes)
    try:
        model.load_state_dict(torch.load(MODELS_DIR / f'{name}_best.pt', map_location=device))
        model = model.to(device)
        model.eval()
        models[name] = model
        print(f"Loaded {name}")
    except:
        print(f"Warning: Could not load {name}")

print(f"\nLoaded {len(models)} models")

## 5.2 Prediction Function

In [ ]:
def predict_ensemble(file_path, models, device, sample_rate=32000, segment_duration=5):
    """Make ensemble predictions for a soundscape file."""
    try:
        waveform, sr = torchaudio.load(file_path)
        if sr != sample_rate:
            resampler = torchaudio.transforms.Resample(sr, sample_rate)
            waveform = resampler(waveform)
        
        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        total_duration = waveform.shape[1] / sample_rate
        num_segments = int(total_duration // segment_duration)
        
        predictions = []
        
        for seg_idx in range(num_segments):
            start = seg_idx * segment_duration * sample_rate
            end = start + segment_duration * sample_rate
            segment = waveform[:, start:end]
            
            # Pad if needed
            if segment.shape[1] < segment_duration * sample_rate:
                segment = torch.nn.functional.pad(segment, (0, segment_duration * sample_rate - segment.shape[1]))
            
            # Ensemble prediction
            seg_preds = []
            for model in models.values():
                with torch.no_grad():
                    output = model(segment.to(device))
                    seg_preds.append(torch.sigmoid(output).cpu().numpy().flatten())
            
            predictions.append(np.mean(seg_preds, axis=0))
        
        return np.array(predictions)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print("Prediction function defined")

## 5.3 Parse Sample Submission

In [ ]:
# Parse sample submission to get expected test files and segments
sample_sub['file_id'] = sample_sub['row_id'].str.rsplit('_', n=1).str[0]
sample_sub['segment_time'] = sample_sub['row_id'].str.rsplit('_', n=1).str[-1].astype(int)

unique_files = sample_sub['file_id'].unique()
print(f"Expected test files: {len(unique_files)}")
print(f"Segments per file: {len(sample_sub) // len(unique_files)}")
print(f"\nSample row_ids:")
print(sample_sub[['row_id', 'file_id', 'segment_time']].head())

## 5.4 Process Test Soundscapes

In [ ]:
# Check for test soundscapes
test_files = list(TEST_SOUNDSCAPES_DIR.glob('*.ogg'))
print(f"Found {len(test_files)} test files in {TEST_SOUNDSCAPES_DIR}")

if len(test_files) == 0:
    print("\nNo test files available yet - they will be populated at submission time.")
    print("Creating submission with placeholder values for demonstration...")
    
    # Create placeholder submission
    submission = sample_sub.copy()
    species_cols = [c for c in submission.columns if c != 'row_id']
    
    # Use baseline prediction (uniform probability)
    baseline_prob = 1.0 / num_classes
    for col in species_cols:
        submission[col] = baseline_prob
    
    print(f"Created baseline submission with uniform probability: {baseline_prob:.6f}")
else:
    print(f"Processing {len(test_files)} test files...")
    
    # Make predictions for all test files
    all_predictions = {}
    
    for file_path in test_files:
        print(f"  Processing: {file_path.name}")
        predictions = predict_ensemble(str(file_path), models, device)
        if predictions is not None:
            all_predictions[file_path.stem] = predictions
    
    print(f"\nProcessed {len(all_predictions)} files")

## 5.5 Create Submission DataFrame

In [ ]:
# Create submission dataframe
if len(test_files) == 0:
    # Use baseline
    submission = sample_sub.copy()
    species_cols = [c for c in submission.columns if c != 'row_id']
    baseline_prob = 1.0 / num_classes
    
    # Add some variation based on training data patterns
    # Get species frequencies from training
    train_df = pd.read_csv(BASE_DIR / 'train.csv')
    species_freq = train_df['primary_label'].value_counts(normalize=True)
    
    for col in species_cols:
        if col in species_freq.index:
            submission[col] = species_freq[col] * 0.1 + baseline_prob * 0.9
        else:
            submission[col] = baseline_prob
    
    print("Created baseline submission with frequency-based priors")
else:
    # Map predictions to submission format
    submission_data = []
    species_cols = species_list
    
    for _, row in sample_sub.iterrows():
        file_id = row['file_id']
        segment_time = row['segment_time']
        
        # Find matching prediction
        if file_id in all_predictions:
            seg_idx = segment_time // 5 - 1  # Convert end_time to index
            if seg_idx >= 0 and seg_idx < len(all_predictions[file_id]):
                pred = all_predictions[file_id][seg_idx]
            else:
                pred = np.ones(num_classes) / num_classes
        else:
            pred = np.ones(num_classes) / num_classes
        
        submission_data.append({
            'row_id': row['row_id'],
            **{species_cols[i]: pred[i] for i in range(num_classes)}
        })
    
    submission = pd.DataFrame(submission_data)
    print(f"Created submission with predictions for {len(submission)} rows")

## 5.6 Validate Submission Format

In [ ]:
# Validate submission format
print("Validating submission format...")

# Check row_ids match
assert len(submission) == len(sample_sub), "Row count mismatch!"
assert all(submission['row_id'] == sample_sub['row_id']), "Row IDs don't match!"
print("✓ Row IDs match")

# Check column order matches
assert list(submission.columns) == list(sample_sub.columns), "Column order mismatch!"
print("✓ Column order matches")

# Check probability values
species_cols = [c for c in submission.columns if c != 'row_id']
probs = submission[species_cols].values
print(f"✓ Probability range: [{probs.min():.6f}, {probs.max():.6f}]")
print(f"✓ Mean probability: {probs.mean():.6f}")

# Check each row sums approximately to 1 (not exactly due to multi-label)
print(f"✓ Sum per row range: [{probs.sum(axis=1).min():.2f}, {probs.sum(axis=1).max():.2f}]")

## 5.7 Save Submission

In [ ]:
# Save submission
submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")

# Show submission preview
print("\nSubmission preview:")
print(submission.head(3))
print(f"\nSubmission shape: {submission.shape}")

## 5.8 Summary Statistics

In [ ]:
# Print summary
print("="*60)
print("SUBMISSION SUMMARY")
print("="*60)

print(f"\nDataset:")
print(f"  Number of test soundscapes: {len(unique_files)}")
print(f"  Number of segments: {len(submission)}")
print(f"  Number of species to predict: {num_classes}")

print(f"\nModel Ensemble:")
for result in training_results:
    print(f"  {result['model']}: Best F1 = {result['best_val_f1']:.4f}")

print(f"\nPrediction Statistics:")
species_cols = [c for c in submission.columns if c != 'row_id']
pred_matrix = submission[species_cols].values
print(f"  Min probability: {pred_matrix.min():.6f}")
print(f"  Max probability: {pred_matrix.max():.6f}")
print(f"  Mean probability: {pred_matrix.mean():.6f}")
print(f"  Std probability: {pred_matrix.std():.6f}")

print(f"\nOutput file: {submission_path}")
print("\n✅ Submission generation complete!")